# PISM-Cloud

PISM-Cloud is a custom AWS deployment of the Alaska Satellite Facilities' serverless, batch processing system, HyP3. PISM-Cloud allows you to run PISM simulations of Alaska glaciers by just providing a RGI glacier complex ID, PISM configuration files, and a run template.

## Connect to PISM-Cloud

In order to equitably distribute PISM-Cloud resources, users must connect to the API with NASA Earthdata Login credentials and be assigned simulation credits.

[Earthdata Login](https://urs.earthdata.nasa.gov/ "https://urs.earthdata.nasa.gov/" )
(EDL) is the authentication method used across NASA's [Earth Observation System Data Information System (EOSDIS)](https://www.earthdata.nasa.gov/about/esdis/eosdis "www.earthdata.nasa.gov/about/esdis/eosdis" ). These credentials allow you to access to any of the Earth Science data products served by EOSDIS, and for our purposes, PISM-Cloud.

There is no cost to [register for EDL credentials](https://urs.earthdata.nasa.gov/users/new "https://urs.earthdata.nasa.gov/users/new" ), and the process is quick and easy. When creating your profile, make sure to select an item in the **Study Area** field, as you may encounter access errors if that field is left blank.

You can connect to PISM-Cloud using the HyP3 SDK like:

In [ ]:
import hyp3_sdk as sdk

pism_cloud = sdk.HyP3('https://pism-cloud-test.asf.alaska.edu', prompt='password')

and check the number of credits you have with:

In [ ]:
pism_cloud.check_credits()

In [ ]:
import fsspec
import s3fs
import geopandas as gpd

In [ ]:
rgi_id = 'RGI2000-v7.0-C-01-04374'

In [ ]:
name = "summer_school"
bucket = "pism-cloud-data"
bucket_prefix = "{name}/test_run/{job_id}"

## Preparing ensemble simulations

Simulation "jobs" are requested in two steps. First, we need to prepare all the necessary input data for the glacier complexes we want to simulate by submitting a PISM_TERRA_PREP_ENSEMBLE job using the template below. These jobs will create the simulation grids; project the input DEM, surface elevations, velocities, etc., onto the grids; and generate executable scripts that will run the simulation.

In [ ]:
# Upload PISM run artifacts to S3.
#
# Three file pickers — config, template, UQ — each immediately uploads the
# selected file to ``s3://{bucket}/glacier/{name}/<kind>/<filename>`` and
# binds the resulting URI to the notebook variables ``pism_config``,
# ``pism_template``, and ``pism_uq`` respectively.

import boto3
import ipywidgets as widgets
from IPython.display import display

# These should already be defined earlier in the notebook (the bucket you
# uploaded to and the study/glacier name). Fallback defaults keep the cell
# runnable in isolation.
bucket = globals().get("bucket", "pism-cloud-data")
name = globals().get("name", "default")

pism_config: str | None = None
pism_template: str | None = None
pism_uq: str | None = None

_s3 = boto3.client("s3")


def _make_uploader(kind: str, var_name: str, accept: str = "") -> widgets.VBox:
    """
    Build a FileUpload widget that ships the dropped file to S3.

    Parameters
    ----------
    kind : str
        Sub-directory under ``s3://{bucket}/glacier/{name}/`` (e.g. ``"config"``).
    var_name : str
        Name of the notebook variable to bind to the resulting ``s3://…`` URI.
    accept : str, optional
        Comma-separated list of accepted file extensions for the picker
        (e.g. ``".toml,.yaml"``). Empty string accepts any file.
    """
    header = widgets.HTML(value=f"<b>Upload {kind} file</b>")
    picker = widgets.FileUpload(accept=accept, multiple=False, description=f"Browse {kind}…")
    status = widgets.HTML(value="<i>no file uploaded yet</i>")

    def _on_change(_change):
        if not picker.value:
            return
        # ipywidgets ≥ 8 returns a tuple of dicts; older versions return a dict.
        item = picker.value[0] if isinstance(picker.value, (list, tuple)) else next(iter(picker.value.values()))
        filename = item.get("name") or item["metadata"]["name"]
        raw = item["content"] if isinstance(item.get("content"), bytes) else bytes(item["content"])
        key = f"glacier/{name}/{kind}/{filename}"
        _s3.put_object(Bucket=bucket, Key=key, Body=raw)
        uri = f"s3://{bucket}/{key}"
        globals()[var_name] = uri
        status.value = f"✅ uploaded → <code>{uri}</code>"

    picker.observe(_on_change, names="value")
    return widgets.VBox([header, picker, status])


display(
    widgets.VBox(
        [
            _make_uploader("config", "pism_config", accept=".toml"),
            _make_uploader("template", "pism_template", accept=".j2,.jinja,.jinja2"),
            _make_uploader("uq", "pism_uq", accept=".toml"),
        ]
    )
)


The `pism_config` and `uq_config` files can be either HTTPS URLs or S3 URIs. We've provided some examples in the public `s3://pism-cloud-data` bucket.

In [ ]:
STAGE_TEMPLATE =     {
    "job_type": "PISM_TERRA_PREP_ENSEMBLE",
    "bucket": bucket,
    "bucket_prefix": bucket_prefix,
    "job_parameters": {
        "pism_config": pism_config,
        "uq_config": pism_uq,
        "ntasks": 32,
    }
}

Then we'll loop through all the RGIs we want to simulate, set the `rgi_id` job parameter, and give the job an appropriate name so we can easily find it later.

In [ ]:
from copy import deepcopy
from pathlib import Path

prepared_jobs = []
for rgi_id in [rgi_id]:
    job_dict = deepcopy(STAGE_TEMPLATE)
    job_dict['job_parameters']['rgi_id'] = rgi_id
    job_dict['name'] = f'{rgi_id}_{Path(job_dict["job_parameters"]["pism_config"]).stem}'
    prepared_jobs.append(job_dict)

In [ ]:
jobs = pism_cloud.submit_prepared_jobs(prepared_jobs)
print(jobs)

job_names = {job.name for job in jobs}
print(job_names)

We can use the `watch` method to periodically check on our jobs and tell us when they have finished.

In [ ]:
jobs = pism_cloud.watch(jobs)

## Execute the PISM-Cloud simulation

The prepare jobs will have created a set of simulations run scripts, which we need to find and then tell PISM-Cloud to execute. All job outputs are uploaded to a "content" bucket under a `job_id/rgi_id` prefix (folder). This function will look in the prefix for all run scripts and return a list of their paths.

In [ ]:
import s3fs

print(jobs)
print(jobs[0])
PISM_CLOUD_BUCKET = jobs[0].files[0]['s3']['bucket']
print(PISM_CLOUD_BUCKET)

def get_run_scripts(job: sdk.Job) ->  list[str]:
    fs = s3fs.S3FileSystem(anon=True)
    files = fs.ls(f'{PISM_CLOUD_BUCKET}/{job.job_id}/{job.job_parameters["rgi_id"]}/run_scripts')
    return [str(Path(file).relative_to(f'{PISM_CLOUD_BUCKET}/{job.job_id}/')) for file in files]


And now, using the execute template, we can run each simulation in the emsemble.

In [ ]:
EXECUTE_TEMPLATE = {
    # "name": "RGI2000-v7.0-C-01-09429_era5_agu_1year",
    "job_type": "PISM_TERRA_EXECUTE",
    "job_parameters": {
        # "ensemble_job_id": "042ffcdc-2134-4b18-b1af-b22fdf7cbb52",
        # "run_script": "RGI2000-v7.0-C-01-09429/run_scripts/submit_g400m_RGI2000-v7.0-C-01-09429_id_0_1978-01-01_1979-01-01.sh",
    }
}

prepared_jobs = []
for job in jobs:
    run_scripts = get_run_scripts(job)
    for script in run_scripts:
        print(script)
        job_dict = deepcopy(EXECUTE_TEMPLATE)
        job_dict['name'] = job.name
        job_dict['job_parameters']['ensemble_job_id'] = job.job_id
        job_dict['job_parameters']['run_script'] = script
        prepared_jobs.append(job_dict)

In [ ]:
jobs = pism_cloud.submit_prepared_jobs(prepared_jobs)
print(jobs)

job_names = {job.name for job in jobs}
print(job_names)

In [ ]:
jobs = pism_cloud.watch(jobs)

In [ ]:
print(jobs)

In [ ]:
import re
import numpy as np
import xarray as xr
from collections import OrderedDict

def preprocess_config(
    ds,
    exp_regexp: str = "id_(.+?)_",
    rgi_regexp: str = r"(RGI2000-v7\.0-C-[^/\s]+)",
    exp_dim: str = "exp_id",
    rgi_dim: str = "rgi_id",
    drop_vars: list[str] | None = None,
    drop_dims: list[str] = ["nv4"],
) -> xr.Dataset:
    """
    Add experiment identifier to the dataset.

    This function processes the dataset by extracting an experiment identifier from the filename
    using a regular expression, adding it as a new dimension, and optionally dropping specified
    variables and dimensions from the dataset.

    Parameters
    ----------
    ds : xarray.Dataset
        The input dataset to be processed.
    exp_regexp : str, optional
        The regular expression pattern to extract the experiment identifier from the filename, by default "id_(.+?)_".
    rgi_regexp : str, optional
        The regular expression pattern to extract the RGI identifier from the filename, by default ``r"(RGI2000-v7\\.0-C-[^/\\s]+)"``.
    exp_dim : str, optional
        The name of the new experiment dimension to be added to the dataset, by default "exp_id".
    rgi_dim : str, optional
        The name of the new RGI dimension to be added to the dataset, by default "rgi_id".
    drop_vars : list[str]| None, optional
        A list of variable names to be dropped from the dataset, by default None.
    drop_dims : list[str], optional
        A list of dimension names to be dropped from the dataset, by default ["nv4"].

    Returns
    -------
    xarray.Dataset
        The processed dataset with the experiment identifier added as a new dimension, and specified variables and dimensions dropped.

    Raises
    ------
    AssertionError
        If the regular expression does not match any part of the filename.
    """

    m_rgi_id_re = re.search(rgi_regexp, ds.command)
    assert m_rgi_id_re is not None
    m_rgi_id = m_rgi_id_re.group(1)

    m_exp_id_re = re.search(exp_regexp, ds.encoding["source"])
    assert m_exp_id_re is not None
    m_exp_id = m_exp_id_re.group(1)

    p_config = ds["pism_config"]
    ds = ds.expand_dims({rgi_dim: [m_rgi_id], exp_dim: [m_exp_id]})

    # List of suffixes to exclude
    suffixes_to_exclude = ["_doc", "_type", "_units", "_option", "_choices"]

    # Filter the dictionary
    config = {k: v for k, v in p_config.attrs.items() if not any(k.endswith(suffix) for suffix in suffixes_to_exclude)}
    if "geometry.front_retreat.prescribed.file" not in config.keys():
        config["geometry.front_retreat.prescribed.file"] = "false"

    config_sorted = OrderedDict(sorted(config.items()))

    pc_keys = np.array(list(config_sorted.keys()))
    pc_vals = np.array(list(config_sorted.values()))

    pism_config = xr.DataArray(
        pc_vals.reshape(-1),
        dims=["pism_config_axis"],
        coords={"pism_config_axis": pc_keys},
        name="pism_config",
    )

    ds = xr.merge(
        [
            ds.drop_vars(["pism_config"], errors="ignore").drop_dims(["pism_config_axis"], errors="ignore"),
            pism_config,
        ]
    )
    return ds.drop_vars(drop_vars, errors="ignore").drop_dims(drop_dims, errors="ignore")


In [ ]:
scalar_files = ["s3://" + f for f in fs.glob(f"{PISM_CLOUD_BUCKET}/**/output/**/fldsum_spatial_*.nc")]

In [ ]:
time_coder = xr.coders.CFDatetimeCoder()
delta_coder = xr.coders.CFTimedeltaCoder()
ds = xr.open_mfdataset(scalar_files, preprocess=preprocess_config,
                       parallel=True,
                       decode_times=time_coder,
                       decode_timedelta=delta_coder,
                       engine="h5netcdf",
                       storage_options={'anon': True})

In [ ]:
ds